# Lecture 2 (optional): finding structure in wine chemistry

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Edinburgh-Chemistry-Teaching/AdvancedPhysicalChemistry/blob/main/L2_unsupervised/L2_unsupervised_wine.ipynb)

<a rel="license" href="https://creativecommons.org/licenses/by-sa/4.0/"><img alt="Creative Commons Licence" style="border-width:0" src="https://licensebuttons.net/l/by-sa/4.0/88x31.png" title='This work is licensed under a Creative Commons Attribution-ShareAlike 4.0 International License.' align="right"/></a>

Author: Dr Antonia Mey
Email: antonia.mey@ed.ac.uk

© Antonia Mey, University of Edinburgh, 2026. Text CC BY-SA 4.0; code cells MIT (see `LICENSE` and `LICENSE-CODE` in the repository).

**Time needed:** about 30–40 minutes. This notebook is **optional** practice for Lecture 2 (unsupervised learning).

### Learning objectives
By the end of this notebook, you should be able to
* Describe a data set as a data matrix of samples × features, and say why features must be scaled before PCA, t-SNE or UMAP.
* Carry out PCA with NumPy (centre, covariance matrix, eigenvalues and eigenvectors, projection) and check it against `scikit-learn`.
* Read a variance-explained (scree) plot and the loadings of a principal component.
* Make t-SNE and UMAP maps, and see how they change with their settings (perplexity, `n_neighbors`, `min_dist`) and the random seed.
* Compare what PCA, t-SNE and UMAP keep (neighbours or distances), and explain what a t-SNE or UMAP map can and cannot tell you.

### Table of Contents

0. [The data: features and labels](#0-data)
1. [Why we scale the features](#1-scaling)
2. [PCA by hand](#2-pca)
3. [Looking at the principal components](#3-pc-plot)
4. [t-SNE: keep neighbours together](#4-tsne)
5. [UMAP: a neighbour graph laid out in 2D](#5-umap)
6. [PCA, t-SNE and UMAP side by side](#6-compare)
7. [Key points](#7-key-points)

### The data

You met this data set in Data-Driven Chemistry: 178 Italian wines from **three grape cultivars**, grown in the same region. For each wine, **13 chemical properties** were measured, for example alcohol content, malic acid, magnesium, total phenols, flavanoids, colour intensity and proline (an amino acid). The data come with `scikit-learn` (`sklearn.datasets.load_wine`; original source: UCI Machine Learning Repository), so nothing needs to be downloaded.

**The idea of this notebook:** we pretend we do *not* know which cultivar each wine is. Can PCA, t-SNE and UMAP find the three groups from the chemistry alone? Only at the end of each section do we use the cultivar labels to check.

### How to open this notebook

* **Noteable (recommended):** open it from the Lecture 2 page on Learn. Everything you need is already installed.
* **Google Colab (backup):** click the *Open in Colab* button at the top. You need a Google account. Save a copy to your Drive (*File → Save a copy in Drive*) or your answers will be lost.
* **Your own computer:** you need `numpy`, `pandas`, `matplotlib`, `scikit-learn` and `umap-learn` (see `requirements.txt` in the repository).

**<span style="color:black">Jupyter Cheat Sheet</span>**
- To run the currently highlighted cell and move focus to the next cell, hold <kbd>&#x21E7; Shift</kbd> and press <kbd>&#x23ce; Enter</kbd>;
- To get help for a specific function, place the cursor within the function's brackets, hold <kbd>&#x21E7; Shift</kbd>, and press <kbd>&#x21E5; Tab</kbd>;
- Cells marked `# FIXME` are for you to complete.

# Import libraries

In [ ]:
# UMAP is not part of scikit-learn: install it if it is missing (e.g. on Colab)
try:
    import umap
except ImportError:
    %pip install -q umap-learn
    import umap

In [ ]:
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.datasets import load_wine
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.manifold import TSNE

warnings.filterwarnings("ignore", message=".*n_jobs.*")   # UMAP: harmless note about random_state

COLOURS = np.array(["#2a78d6", "#eb6834", "#2f8f5b"])   # one colour per group

# 0. The data: features and labels <a id="0-data"></a>

Each row of the table is one **sample** (a wine); each column is one **feature** (a measured property). Together they form the **data matrix** $X$ with $N$ rows and $d$ columns. The cultivar is a **label**: we keep it separate and only use it to check our results.

In [ ]:
wine = load_wine(as_frame=True)
X_df = wine.data                  # features: 13 chemical properties
cultivar = wine.target.values     # label: 0, 1 or 2 (kept aside!)

print("data matrix X:", X_df.shape, "(N samples, d features)")
X_df.head()

In [ ]:
# summary of each feature: note the very different ranges and units
X_df.describe().T[["mean", "std", "min", "max"]]

# 1. Why we scale the features <a id="1-scaling"></a>

PCA looks for the directions of **largest variance**, and t-SNE and UMAP look for **nearest neighbours**, i.e. they measure distances. Both are dominated by whichever feature has the largest numbers. Proline is measured in mg/L and varies by hundreds; hue varies by a fraction of 1. Without scaling, "the direction of largest variance" would simply be "the proline axis".

We therefore **standardise** every feature: subtract its mean and divide by its standard deviation,
$$z_{ij} = \frac{x_{ij} - \bar x_j}{\sigma_j}.$$
Every feature then has mean 0 and variance 1. (PCA on standardised data is the same as PCA on the *correlation* matrix instead of the covariance matrix.)

## Tasks 1 <a id="tasks-1"></a>
1. Compute the variance of each feature in `X_df`. Which feature has the largest variance, and roughly how much larger is it than the smallest?
2. The cell below standardises the data with `StandardScaler` and stores it as `Z`. Check that every column of `Z` now has mean ≈ 0 and variance ≈ 1.

In [ ]:
Z = StandardScaler().fit_transform(X_df)   # standardised data matrix, used from now on

In [ ]:
# 1. Variance of each feature
# FIXME

In [ ]:
# 2. Check the standardised data
# FIXME

# 2. PCA by hand <a id="2-pca"></a>

These are exactly the steps from the board in Lecture 2, now for 13 features instead of 2:

1. **Centre** the data (already done by standardising).
2. Compute the **covariance matrix** $\mathbf C = \frac{1}{N}\mathbf Z^\mathsf{T}\mathbf Z$ (a $d \times d$ matrix).
3. Find its **eigenvalues** $\lambda_k$ and **eigenvectors** $\mathbf v_k$: $\mathbf C\mathbf v_k = \lambda_k \mathbf v_k$. Sort them from the largest eigenvalue down. The eigenvectors are the principal components (PCs); each eigenvalue is the variance along its PC.
4. **Fraction of variance explained** by PC $k$: $\lambda_k / \sum_j \lambda_j$.
5. **Project** each sample onto the PCs: the *scores* are $\mathbf Z \mathbf V$.

`np.linalg.eigh` is the NumPy function for eigenvalues of a symmetric matrix; it returns them in *ascending* order.

In [ ]:
N = Z.shape[0]
C = Z.T @ Z / N                       # 13 x 13 covariance matrix
eigvals, eigvecs = np.linalg.eigh(C)  # ascending order
order = np.argsort(eigvals)[::-1]     # largest first
eigvals, eigvecs = eigvals[order], eigvecs[:, order]

print("C has shape", C.shape)
print("eigenvalues:", np.round(eigvals, 2))

## Tasks 2 <a id="tasks-2"></a>
1. Compute the fraction of variance explained by each PC and the cumulative fraction. Plot both against the PC number (a *scree plot*).
2. How many PCs do you need to keep 90 % of the variance? What fraction do PC1 and PC2 explain together?
3. The cell below projects the data onto the PCs (`scores = Z @ eigvecs`) and keeps the first two columns as `pc_scores`. Check your scores against `PCA(n_components=2).fit_transform(Z)`. (The signs of whole columns may be flipped: an eigenvector multiplied by −1 is still an eigenvector.)

In [ ]:
scores = Z @ eigvecs          # projection onto all 13 PCs
pc_scores = scores[:, :2]       # keep PC1 and PC2

In [ ]:
# 1. Variance explained and scree plot
# FIXME

In [ ]:
# 2. How many PCs for 90 %? PC1 + PC2?
# FIXME

In [ ]:
# 3. Compare with scikit-learn
# FIXME

# 3. Looking at the principal components <a id="3-pc-plot"></a>

We have reduced every wine from 13 numbers to 2. First, look at the result **without** the labels: this is all an unsupervised method sees.

In [ ]:
fig, ax = plt.subplots(figsize=(5, 5))
ax.scatter(pc_scores[:, 0], pc_scores[:, 1], s=20, color="grey")
ax.set_xlabel("PC1"); ax.set_ylabel("PC2")
ax.set_title("178 wines, no labels")
plt.show()

Now we reveal the cultivars (only to check what PCA found):

In [ ]:
fig, ax = plt.subplots(figsize=(5, 5))
for c in range(3):
    m = cultivar == c
    ax.scatter(pc_scores[m, 0], pc_scores[m, 1], s=20, color=COLOURS[c], label=f"cultivar {c}")
ax.set_xlabel("PC1"); ax.set_ylabel("PC2"); ax.legend()
plt.show()

The **loadings** of a PC are the entries of its eigenvector: how much each original feature contributes to it. A large positive or negative loading means that feature matters a lot along that PC.

## Tasks 3 <a id="tasks-3"></a>
1. Make a table (or bar chart) of the loadings of PC1, i.e. the first column of `eigvecs`, with the feature names. Which properties dominate PC1? Which have the opposite sign?
2. Chemically, what might PC1 be measuring? (Hint: flavanoids and proanthocyanins are both phenolic compounds.)
3. Do the three cultivars look separable in the PC1–PC2 plane? Which two overlap most?

In [ ]:
# 1. Loadings of PC1
# FIXME

*Your answers to questions 2 and 3 here.*

# 4. t-SNE: keep neighbours together <a id="4-tsne"></a>

PCA is **linear**: it can only rotate the data and drop directions. t-SNE (Lecture 2) is **non-linear**. The idea:

1. In the 13-dimensional space, work out for every wine which other wines are its **neighbours** (as probabilities; the *perplexity* sets roughly how many neighbours count).
2. Start with every wine at a random position in 2D.
3. Move the 2D points by **gradient descent** (as in Lecture 1) until the neighbours in 2D match the neighbours in 13D.

Neighbours are pulled together; wines that were far apart can end up anywhere. So t-SNE is good at showing *which samples belong together*, but **the sizes of the groups, the distances between groups and the axes have no meaning**. A good, interactive read: [How to use t-SNE effectively](https://distill.pub/2016/misread-tsne/) (distill.pub).

As in Section 3, we first look **without** the labels.

In [ ]:
Y_tsne = TSNE(n_components=2, perplexity=30, random_state=0).fit_transform(Z)

fig, ax = plt.subplots(figsize=(5, 5))
ax.scatter(Y_tsne[:, 0], Y_tsne[:, 1], s=20, color="grey")
ax.set_xticks([]); ax.set_yticks([])          # t-SNE axes have no meaning
ax.set_title("t-SNE (perplexity 30), no labels")
plt.show()

## Tasks 4 <a id="tasks-4"></a>
1. Plot `Y_tsne` again, now coloured by cultivar (as in Section 3). Are the three cultivars separated?
2. **Perplexity:** run t-SNE with perplexity 5, 30 and 50 and plot the three maps side by side, coloured by cultivar. What changes, and what stays the same?
3. **Pitfalls:** run t-SNE (perplexity 30) twice with `init="random"`, once with `random_state=0` and once with `random_state=1`, and plot both. Then answer:
   * Is the map the same both times? Does it matter that the groups are in different places or rotated?
   * In the PCA plot of Section 3, the three cultivars have different spreads. Is that still true in the t-SNE map? What does that tell you about reading the size of a group, or the gap between two groups, off a t-SNE map?

In [ ]:
# 1. t-SNE coloured by cultivar
# FIXME

In [ ]:
# 2. Three perplexities
# FIXME

In [ ]:
# 3. Two random seeds
# FIXME

*Your answers to question 3 here.*

# 5. UMAP: a neighbour graph laid out in 2D <a id="5-umap"></a>

UMAP has the same aim as t-SNE but works with a **graph**:

1. Join every sample to its `n_neighbors` nearest neighbours in 13D.
2. Lay this graph out in 2D so that connected samples stay close. `min_dist` sets how tightly points may pack together.

It is usually faster than t-SNE and keeps a bit more of the large-scale layout, but the same warning applies: **group sizes, gaps and axes have little meaning**.

In [ ]:
Y_umap = umap.UMAP(n_neighbors=15, min_dist=0.1, random_state=0).fit_transform(Z)

## Tasks 5 <a id="tasks-5"></a>
1. Plot `Y_umap` coloured by cultivar.
2. Make a grid of UMAP maps for `n_neighbors` = 5, 15, 50 (columns) and `min_dist` = 0.0, 0.5 (rows), all with `random_state=0`. How does each parameter change the picture?

In [ ]:
# 1. UMAP coloured by cultivar
# FIXME

In [ ]:
# 2. Grid of n_neighbors and min_dist
# FIXME

# 6. PCA, t-SNE and UMAP side by side <a id="6-compare"></a>

What does each map keep? We can measure two things for every map:

* **Neighbours kept (local):** for each wine, which fraction of its 10 nearest neighbours in 13D are also among its 10 nearest neighbours in the 2D map?
* **Distances kept (global):** how well do the distances between *all* pairs of wines in the map follow the distances in 13D? (Spearman rank correlation, 1 = same order.)

The function below computes both.

In [ ]:
from scipy.spatial.distance import pdist
from scipy.stats import spearmanr
from sklearn.neighbors import NearestNeighbors


def what_is_kept(Y, X=Z, k=10):
    """Return (fraction of k nearest neighbours kept, rank correlation of all pairwise distances)."""
    nn_X = NearestNeighbors(n_neighbors=k + 1).fit(X).kneighbors(X, return_distance=False)[:, 1:]
    nn_Y = NearestNeighbors(n_neighbors=k + 1).fit(Y).kneighbors(Y, return_distance=False)[:, 1:]
    neighbours = np.mean([len(set(a) & set(b)) / k for a, b in zip(nn_X, nn_Y)])
    distances = spearmanr(pdist(X), pdist(Y))[0]
    return neighbours, distances

## Tasks 6 <a id="tasks-6"></a>
1. Plot the PCA, t-SNE and UMAP maps (`pc_scores`, `Y_tsne`, `Y_umap`) side by side, coloured by cultivar.
2. Use `what_is_kept` on each map and print the two numbers. Which method keeps neighbours best? Which keeps the overall distances best?
3. **Reflection:** a colleague shows you a t-SNE map of 2,000 IR spectra with two well-separated islands, one twice as big as the other, and concludes: "there are two classes of compound, and the first is much more varied". What can you conclude from such a map, what can you not, and what would you check?

In [ ]:
# 1. Three maps side by side
# FIXME

In [ ]:
# 2. What does each map keep?
# FIXME

*Your answer to question 3 here.*

# 7. Key points <a id="7-key-points"></a>

* Unsupervised learning works on the data matrix $X$ alone; labels, if we have them, are only for checking.
* **Scale your features** before PCA, t-SNE or UMAP, otherwise the feature with the largest numbers dominates.
* **PCA** = eigenvectors of the covariance matrix; each eigenvalue is the variance along its PC. A few PCs often capture most of the variance (here PC1 and PC2: about 55 %). PCA is linear and keeps the overall distances.
* The **loadings** tell you which original features make up a PC, and often have a chemical meaning.
* **t-SNE** and **UMAP** are non-linear maps that keep **neighbours** together. They can separate groups that PCA overlaps, but group sizes, gaps and axes mean little, and the picture depends on the settings and the seed.
* Use t-SNE and UMAP for looking at data, not as evidence on their own.

**Questions to think about:**
1. Why can PCA show three clear groups even though it never saw the cultivar labels?
2. If t-SNE shows a fourth small group in new wine data, would you believe there is a fourth cultivar? What would you check?